# Open Insurance Demo — Centralized MCP with Auth Propagation

This notebook demonstrates **centralized MCP server management** with **Okta OIDC auth propagation** using AWS AgentCore Gateway — themed for an **insurance** use case.

## The "whoa" moments

1. **One Gateway endpoint** manages both Lambda functions AND MCP servers
2. **User identity flows through** — each user's JWT is validated by the Gateway
3. **Different users, different tools** — Alice (customer service) sees only policy lookup; Bob (claims adjuster) sees both
4. **Gateway-level ENFORCE** — Cedar policies block unauthorized tool calls at the Gateway, not the agent

## Insurance Demo Personas

| User | Role | Tools Available |
|------|------|----------------|
| Alice | Customer Service Rep | `PolicyLookup___lookup_policy` |
| Bob | Claims Adjuster | `PolicyLookup___lookup_policy` + `ClaimsData___query_claims` |

## Prerequisites

Run `01_setup.ipynb` first to create the infrastructure.

## Cell 1: Load Config + Pre-Authenticate Both Users

We authenticate both Alice and Bob upfront via Okta Resource Owner Password flow, then create separate Strands agents for each — one connected to the Gateway with Alice's token, one with Bob's.

The Gateway validates each JWT against Okta's JWKS endpoint and checks:
- **Audience** matches `api://default`
- **Client ID** matches the configured `allowedClients`
- **Signature** is valid per Okta's signing keys

Cedar policies in **ENFORCE mode** then control which tools each user can discover and invoke:
- `AgentCore::OAuthUser` (all) → PolicyLookup: **ALLOW**
- `AgentCore::OAuthUser::"bob@..."` → ClaimsData: **ALLOW**
- Everyone else → ClaimsData: **DENY** (no matching permit)

In [4]:
import os
import json
import jwt
import httpx
import requests
from dotenv import load_dotenv
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamable_http_client

load_dotenv(override=True)

# --- Load Gateway config from setup notebook ---
with open("gateway_config.json") as f:
    config = json.load(f)

GATEWAY_URL = config["gateway_url"]
OKTA_DOMAIN = config["okta_domain"]
OKTA_CLIENT_ID = config["okta_client_id"]
OKTA_ISSUER = config["okta_issuer"]
OKTA_CLIENT_SECRET = os.environ["OKTA_CLIENT_SECRET"]

# --- Okta credentials for test users ---
ALICE_USERNAME = os.environ["ALICE_USERNAME"]
ALICE_PASSWORD = os.environ["ALICE_PASSWORD"]
BOB_USERNAME = os.environ["BOB_USERNAME"]
BOB_PASSWORD = os.environ["BOB_PASSWORD"]

# --- Model to use for Strands agents ---
# Use AU cross-region inference profile for ap-southeast-2
MODEL_ID = "au.anthropic.claude-sonnet-4-6"


def get_okta_token(username: str, password: str) -> dict:
    """Authenticate a user via Okta Resource Owner Password flow and return the JWT."""
    token_url = f"{OKTA_ISSUER}/v1/token"
    response = requests.post(
        token_url,
        data={
            "grant_type": "password",
            "username": username,
            "password": password,
            "scope": "openid profile groups",
            "client_id": OKTA_CLIENT_ID,
            "client_secret": OKTA_CLIENT_SECRET,
        },
        headers={"Content-Type": "application/x-www-form-urlencoded"},
    )
    response.raise_for_status()
    token_data = response.json()
    access_token = token_data["access_token"]

    # Decode JWT (without verification, just to display claims)
    claims = jwt.decode(access_token, options={"verify_signature": False})
    return {"access_token": access_token, "claims": claims}


# --- Pre-authenticate both users ---
print("Authenticating Alice (customer service rep)...")
alice_auth = get_okta_token(ALICE_USERNAME, ALICE_PASSWORD)
print(f"  User: {alice_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {alice_auth['claims'].get('groups', [])}")

print("\nAuthenticating Bob (claims adjuster)...")
bob_auth = get_okta_token(BOB_USERNAME, BOB_PASSWORD)
print(f"  User: {bob_auth['claims'].get('sub', 'unknown')}")
print(f"  Groups: {bob_auth['claims'].get('groups', [])}")

# --- Create MCPClients connected to Gateway with each user's token ---
alice_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {alice_auth['access_token']}"})
bob_http = httpx.AsyncClient(headers={"Authorization": f"Bearer {bob_auth['access_token']}"})

alice_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=alice_http),
    startup_timeout=60,
)

bob_client = MCPClient(
    lambda: streamable_http_client(GATEWAY_URL, http_client=bob_http),
    startup_timeout=60,
)

# --- Create Strands Agents ---
# No agent-level RBAC needed — Cedar policies in ENFORCE mode handle access control
# at the Gateway level. The agent simply calls tools; the Gateway decides what's allowed.
alice_client.__enter__()
bob_client.__enter__()

alice_tools = alice_client.list_tools_sync()
bob_tools = bob_client.list_tools_sync()

SYSTEM_PROMPT = """You are a helpful assistant for an insurance company.
If a tool call fails with an access denied or policy enforcement error, explain clearly
that the user does not have permission to use that tool."""

alice_agent = Agent(
    model=MODEL_ID,
    tools=alice_tools,
    system_prompt=SYSTEM_PROMPT,
)

bob_agent = Agent(
    model=MODEL_ID,
    tools=bob_tools,
    system_prompt=SYSTEM_PROMPT,
)

alice_tool_names = [t.tool_name if hasattr(t, 'tool_name') else str(t) for t in alice_tools]
bob_tool_names = [t.tool_name if hasattr(t, 'tool_name') else str(t) for t in bob_tools]

print(f"\n--- Gateway Authentication (ENFORCE mode) ---")
print(f"  Gateway URL: {GATEWAY_URL}")
print(f"  Model:       {MODEL_ID}")
print(f"  Alice tools: {alice_tool_names}")
print(f"  Bob tools:   {bob_tool_names}")
print(f"\n  Alice sees {len(alice_tools)} tool(s), Bob sees {len(bob_tools)} tool(s)")
print(f"  Access control enforced by Cedar policies at the Gateway")

Authenticating Alice (customer service rep)...
  User: alice@integrator-5723923.okta.com
  Groups: ['analysts', 'Everyone']

Authenticating Bob (claims adjuster)...
  User: bob@integrator-5723923.okta.com
  Groups: ['finance-admins', 'Everyone']

--- Gateway Authentication (ENFORCE mode) ---
  Gateway URL: https://agentcore-mcp-demo-gateway-r68b4sket5.gateway.bedrock-agentcore.ap-southeast-2.amazonaws.com/mcp
  Model:       au.anthropic.claude-sonnet-4-6
  Alice tools: ['PolicyLookup___lookup_policy']
  Bob tools:   ['ClaimsData___query_claims', 'PolicyLookup___lookup_policy']

  Alice sees 1 tool(s), Bob sees 2 tool(s)
  Access control enforced by Cedar policies at the Gateway


## Cell 2: Show Available Tools Per User

With ENFORCE mode, the Gateway filters tool discovery based on Cedar policies.
Alice (customer service) only sees policy lookup tools; Bob (claims adjuster) sees both policy and claims tools.

In [5]:
print("Tools visible through AgentCore Gateway:\n")

print("ALICE (customer service rep):")
for tool in alice_tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")
print(f"  Total: {len(alice_tools)} tool(s)\n")

print("BOB (claims adjuster):")
for tool in bob_tools:
    name = tool.tool_name if hasattr(tool, 'tool_name') else str(tool)
    print(f"  - {name}")
print(f"  Total: {len(bob_tools)} tool(s)\n")

print("Cedar ENFORCE mode filters tool discovery per user.")
print("Alice cannot even see ClaimsData — the Gateway hides it from her.")

Tools visible through AgentCore Gateway:

ALICE (customer service rep):
  - PolicyLookup___lookup_policy
  Total: 1 tool(s)

BOB (claims adjuster):
  - ClaimsData___query_claims
  - PolicyLookup___lookup_policy
  Total: 2 tool(s)

Cedar ENFORCE mode filters tool discovery per user.
Alice cannot even see ClaimsData — the Gateway hides it from her.


## Cell 3: Alice Looks Up a Policy (ALLOWED)

Alice is a customer service representative. The Cedar policy allows **all authenticated users** to access `PolicyLookup`. This should succeed — customer service reps need policy details to help callers.

In [6]:
print("=" * 60)
print("ALICE (customer service) asks: 'Look up policy POL-10042'")
print("=" * 60)

response = alice_agent("Can you look up insurance policy POL-10042? I have a customer on the line asking about their coverage.")

print("\n" + "=" * 60)
print("RESULT: Alice (customer service) CAN look up policy details")
print("Cedar policy: PolicyLookup → ALLOW all authenticated users")
print("=" * 60)

ALICE (customer service) asks: 'Look up policy POL-10042'
Sure! Let me look that up right away.
Tool #1: PolicyLookup___lookup_policy
Here are the details for policy **POL-10042**:

- **Policy Holder:** Sarah Chen
- **Policy Type:** Auto Insurance
- **Status:** ✅ Active
- **Vehicle:** 2023 Toyota RAV4
- **Coverage:**
  - Liability: $500,000
  - Collision: $100,000
- **Monthly Premium:** $185.00
- **Effective Date:** June 1, 2025
- **Expiry Date:** June 1, 2026

The policy is currently active and valid through June 2026. Is there anything specific about Sarah's coverage you'd like to discuss or any other questions I can help with?
RESULT: Alice (customer service) CAN look up policy details
Cedar policy: PolicyLookup → ALLOW all authenticated users


## Cell 4: Alice Asks for Claims Data (BLOCKED by Gateway)

Alice asks for confidential claims information. The Cedar policy only permits Bob's `sub` to access ClaimsData.
The Gateway **enforces** this — Alice's agent doesn't even have `ClaimsData___query_claims` in its tool list.

**This is the key "whoa" moment — the Gateway enforces access control, not the LLM.**
A customer service rep should never see adjuster notes, fraud flags, or claim amounts.

In [7]:
print("=" * 60)
print("ALICE (customer service) asks: 'Show me the claims summary'")
print("=" * 60)

response = alice_agent("Can you show me the claims summary? I want to see all recent insurance claims and adjuster notes.")

print("\n" + "=" * 60)
print("RESULT: Alice CANNOT access claims data!")
print(f"Alice's available tools: {alice_tool_names}")
print("Gateway Cedar policy: ClaimsData not in Alice's permitted actions")
print("The Gateway doesn't even expose ClaimsData to Alice's agent")
print("=" * 60)

ALICE (customer service) asks: 'Show me the claims summary'
I appreciate you asking, but unfortunately **I don't have a tool available to look up claims summaries or adjuster notes**. My current capabilities are limited to looking up policy details by policy number.

To access claims information and adjuster notes, I'd suggest:

1. **Internal Claims System** – Log into your company's claims management platform directly for a full claims history and adjuster notes.
2. **Claims Department** – Reach out to your claims team who would have full access to that information.
3. **Supervisor/Admin Access** – If you need elevated access to claims data, your system administrator may be able to assist.

Is there anything else I can help with, such as looking up another policy? I'm happy to assist with what's available to me!
RESULT: Alice CANNOT access claims data!
Alice's available tools: ['PolicyLookup___lookup_policy']
Gateway Cedar policy: ClaimsData not in Alice's permitted actions
The Gatewa

## Cell 5: Bob Asks for Claims Data (ALLOWED by Gateway)

Now Bob asks the **exact same question**. His JWT `sub` matches the Cedar policy for ClaimsData.
The Gateway permits the call and returns the confidential claims information.

**Same question, different user, different result — enforced by Gateway Cedar policies, not the LLM.**
As a claims adjuster, Bob needs to see claim amounts, adjuster notes, and fraud flags to do his job.

In [8]:
print("=" * 60)
print("BOB (claims adjuster) asks: 'Show me the claims summary'")
print("=" * 60)

response = bob_agent("Show me the claims summary. I need to review all recent insurance claims including adjuster notes and investigation status.")

print("\n" + "=" * 60)
print("RESULT: Bob (claims adjuster) CAN access claims data!")
print(f"Bob's available tools: {bob_tool_names}")
print("Same question as Alice. Same Gateway. Same endpoint.")
print("The ONLY difference: Bob's JWT sub matches the Cedar policy")
print("=" * 60)

BOB (claims adjuster) asks: 'Show me the claims summary'
I'll retrieve the claims summary for you right away!
Tool #1: ClaimsData___query_claims
Here is the **Claims Summary** for your review:

---

## 📊 Q1 2026 Quarterly Overview

| Metric | Value |
|---|---|
| Total Claims Filed | 147 |
| Total Claims Resolved | 112 |
| Total Amount Paid | $2.3M |
| Avg. Resolution Time | 18 days |
| Top Category | Auto - Collision (42%) |
| Fraud Flagged | 3 |

---

## 🗂️ Recent Claims Detail

### 1. CLM-2024-0891 — Sarah Chen
- **Type:** Auto - Collision
- **Policy:** POL-10042
- **Status:** ✅ Approved
- **Claimed:** $12,400 | **Approved:** $11,800
- **Filed:** 2025-12-15 | **Resolved:** 2026-01-20
- **Adjuster Notes:** Rear-end collision on Pacific Hwy. Liability confirmed via dashcam footage. Deductible of $600 applied.

---

### 2. CLM-2025-0023 — James Rodriguez
- **Type:** Home - Storm Damage
- **Policy:** POL-10078
- **Status:** 🔄 In Progress
- **Claimed:** $45,000 | **Approved:** Pending
- *

## Cell 6: Architecture Summary

What we just demonstrated — an insurance company's AI agents with role-based access control:

In [9]:
print("""
==================================================================
    Open Insurance — AgentCore Gateway Demo Summary
==================================================================

  ARCHITECTURE:

    User (CLI)
      |
      | username + password (Resource Owner Password grant)
      v
    Okta Token Endpoint --> JWT (with groups, client_id claims)
      |
      v
    Strands Agent (JWT as Bearer token)
      |
      | MCP (StreamableHTTP + Bearer JWT)
      v
    AgentCore Gateway
      |-- Ingress Auth: JWT validation (signature, audience, client_id)
      |-- Cedar Policy Engine (ENFORCE mode)
      |     principal: AgentCore::OAuthUser::"<JWT sub>"
      |     action:    AgentCore::Action::"<TargetName>"
      |     resource:  AgentCore::Gateway::"<gateway-arn>"
      |
      +------------------+
      |                  |
      v                  v
    Lambda            Lambda
  (PolicyLookup)    (ClaimsData)

  AUTH FLOW:
    1. User authenticates via Okta ROPC grant -> JWT with sub claim
    2. Strands Agent attaches JWT as Bearer token on MCP connection
    3. Gateway validates JWT via Okta JWKS endpoint
    4. Cedar policies evaluated in ENFORCE mode
    5. Gateway permits or denies tool discovery and invocation
    6. No access control logic needed in the agent or MCP servers

  CEDAR POLICIES:
    PolicyLookup -> ALL AgentCore::OAuthUser -> ALLOW
    ClaimsData   -> AgentCore::OAuthUser::"bob@..." -> ALLOW
    ClaimsData   -> everyone else -> DENY (no matching permit)

  WHAT WE DEMONSTRATED:
    Alice (customer service)  -> Policy lookup  -> ALLOWED (Gateway permit)
    Alice (customer service)  -> Claims data    -> BLOCKED (Gateway deny)
    Bob (claims adjuster)     -> Claims data    -> ALLOWED (Gateway permit)

  KEY VALUE FOR INSURANCE:
    - ONE Gateway manages all internal insurance tools
    - Direct Okta OIDC -- no Cognito intermediary
    - Cedar policies ENFORCE access at the Gateway level
    - Customer service reps CANNOT see claims adjuster notes or fraud flags
    - Claims adjusters CAN see both policies and confidential claims
    - Agent needs zero access control logic
    - Zero auth code in MCP servers
    - Compliance team can audit Cedar policies directly

==================================================================
""")

# Cleanup MCPClients and httpx clients
alice_client.__exit__(None, None, None)
bob_client.__exit__(None, None, None)
print("Agents disconnected. Run cleanup in 01_setup.ipynb when done.")


    Open Insurance — AgentCore Gateway Demo Summary

  ARCHITECTURE:

    User (CLI)
      |
      | username + password (Resource Owner Password grant)
      v
    Okta Token Endpoint --> JWT (with groups, client_id claims)
      |
      v
    Strands Agent (JWT as Bearer token)
      |
      | MCP (StreamableHTTP + Bearer JWT)
      v
    AgentCore Gateway
      |-- Ingress Auth: JWT validation (signature, audience, client_id)
      |-- Cedar Policy Engine (ENFORCE mode)
      |     principal: AgentCore::OAuthUser::"<JWT sub>"
      |     action:    AgentCore::Action::"<TargetName>"
      |     resource:  AgentCore::Gateway::"<gateway-arn>"
      |
      +------------------+
      |                  |
      v                  v
    Lambda            Lambda
  (PolicyLookup)    (ClaimsData)

  AUTH FLOW:
    1. User authenticates via Okta ROPC grant -> JWT with sub claim
    2. Strands Agent attaches JWT as Bearer token on MCP connection
    3. Gateway validates JWT via Okta JWKS endp

---

## Bonus: Claude Code Live Demo (Interactive)

After running the notebook demo above, you can show the **same access control working inside Claude Code** — the AI coding assistant connected to the same Gateway.

### Setup

Run `03_claude_code_oauth_demo.ipynb` first to create the Okta SPA app and configure Claude Code.

### Step-by-Step

**1. Start Claude Code** (in a terminal, from this project directory):
```bash
cd /path/to/demo-centralized-mcp-server
claude
```

**2. Okta login opens in browser** — log in as **Bob** first (he can see everything):
- Use your Bob credentials from `.env` (`BOB_USERNAME` / `BOB_PASSWORD`)
- You should see: *"Authentication successful. Connected to agentcore-gateway."*

**3. Verify tools** — type `/mcp` in Claude Code to see available tools:
- Bob should see: `PolicyLookup___lookup_policy` + `ClaimsData___query_claims`

**4. Ask natural language questions:**
```
> Look up insurance policy POL-10042
> Show me the claims summary including adjuster notes
> What's the status of claim CLM-2025-0067?
```
Bob gets full access — policy details AND confidential claims data.

**5. Switch to Alice** — clear token and restart:
```bash
# In another terminal:
rm -rf ~/.claude/oauth-tokens/

# Then restart Claude Code:
claude
```

**6. Log in as Alice:**
- Use your Alice credentials from `.env` (`ALICE_USERNAME` / `ALICE_PASSWORD`)

**7. Verify tools** — type `/mcp`:
- Alice should see: `PolicyLookup___lookup_policy` only (no ClaimsData!)

**8. Ask the same questions:**
```
> Look up insurance policy POL-10042       ← WORKS (policy lookup allowed)
> Show me the claims summary               ← BLOCKED (no claims tool visible)
```

### Key Talking Points During Demo

- "Notice Alice doesn't even **see** the claims tool — the Gateway hides it completely"
- "Same Gateway URL, same agent code, same question — the **only** difference is who logged in"
- "The Lambda functions have **zero auth logic** — all access control is in Cedar policies at the Gateway"
- "Your compliance team can read and audit these Cedar policies directly — they're human-readable"
- "Adding a new role? Just add a Cedar policy. No code changes, no redeployment."